In [ ]:
!pip install gymnasium[atari]
!pip install ale-py

In [13]:
import os
import random
import numpy as np
import gymnasium as gym
import cv2
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, Flatten, Dense, Input, Add
from tensorflow.keras.optimizers import Adam
from collections import deque
import gc
import matplotlib.pyplot as plt  # ADICIONADO PARA PLOTAGEM

# --- CONFIGURAÇÃO DO ATARI ---
import ale_py
gym.register_envs(ale_py)

# --- HIPERPARÂMETROS DE ELITE ---
ENV_NAME = "ALE/MsPacman-v5"
DRIVE_PATH = "/content/drive/MyDrive/dueling_ddqn_mspacman.weights.h5"

EPISODE_START = 3100
EPISODES = 6000          # Nova escala de treino para o novo cérebro
BATCH_SIZE = 128         # Mantido o lote robusto que funcionou bem
MEMORY_SIZE = 30000      # Expandido! Aproveitando a Alta RAM do Colab Pro
GAMMA = 0.99
LEARNING_RATE = 0.00025

EPSILON = 0.2            # Reset total do caos para o novo cérebro aprender do zero
EPSILON_MIN = 0.2        # Agora podemos buscar a precisão de 1% no final!
EPSILON_DECAY = 0.999985 # Decaimento suave para os 3000 episódios

print("--- [SISTEMA] CONECTANDO AO GOOGLE DRIVE ---")
from google.colab import drive
drive.mount('/content/drive')

env = gym.make(ENV_NAME, repeat_action_probability=0.25)
action_space_size = int(env.action_space.n)

def preprocess_frame(frame):
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    frame = cv2.resize(frame, (84, 84))
    return (frame / 255.0).astype(np.float32)

# --- ARQUITETURA DUELING NETWORK ---
def build_dueling_dqn(input_shape, action_space):
    inputs = Input(shape=input_shape)

    # Camadas Convolucionais para enxergar o jogo
    x = Conv2D(32, (8, 8), strides=4, activation="relu")(inputs)
    x = Conv2D(64, (4, 4), strides=2, activation="relu")(x)
    x = Conv2D(64, (3, 3), strides=1, activation="relu")(x)
    x = Flatten()(x)

    # Divisão Dueling: Fluxo de Valor (Value)
    value_fc = Dense(512, activation="relu")(x)
    value = Dense(1)(value_fc)

    # Divisão Dueling: Fluxo de Vantagem (Advantage)
    advantage_fc = Dense(512, activation="relu")(x)
    advantage = Dense(action_space)(advantage_fc)

    # Junção Estável: Q(s,a) = V(s) + (A(s,a) - mean(A(s,a)))
    mean_advantage = tf.keras.layers.Lambda(lambda x: tf.reduce_mean(x, axis=1, keepdims=True))(advantage)
    centered_advantage = tf.keras.layers.Subtract()([advantage, mean_advantage])
    outputs = Add()([value, centered_advantage])

    model = Model(inputs=inputs, outputs=outputs)
    return model

# Criamos a Rede Principal e a Rede Alvo
input_shape = (84, 84, 4)
model = build_dueling_dqn(input_shape, action_space_size)
target_model = build_dueling_dqn(input_shape, action_space_size)
target_model.set_weights(model.get_weights())

optimizer = Adam(learning_rate=LEARNING_RATE)
model.compile(optimizer=optimizer, loss="huber")

# Tenta carregar se já houver um treino Dueling anterior salvo
if os.path.exists(DRIVE_PATH):
    model.load_weights(DRIVE_PATH)
    target_model.set_weights(model.get_weights())
    print("--- [SUCESSO] Pesos Dueling carregados do Drive! ---")
else:
    print("--- [INFO] Inicializando novo cérebro Dueling do zero... ---")

@tf.function
def train_step(states, targets):
    with tf.GradientTape() as tape:
        predictions = model(states, training=True)
        loss = model.compute_loss(states, targets, predictions)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

replay_buffer = deque(maxlen=MEMORY_SIZE)

# --- ARMAZENAMENTO HISTÓRICO ---
historico_rewards = []  # ADICIONADO PARA GUARDAR OS DADOS DO GRÁFICO

print("--- [TREINO] Iniciando a Era de Elite (Dueling Double DQN) ---")

for episode in range(EPISODE_START, EPISODES + 1):
    state, info = env.reset()
    state_deque = deque(maxlen=4)
    frame = preprocess_frame(state)
    for _ in range(4):
        state_deque.append(frame)

    done = False
    total_reward = 0
    steps = 0

    # Variáveis do Anti-Travamento (Anti-Wall Stuck)
    last_score_step = 0
    last_reward_total = 0

    while not done:
        steps += 1
        state_stacked = np.stack(state_deque, axis=-1)
        state_input = np.expand_dims(state_stacked, axis=0)

        # Epsilon-Greedy
        if random.random() < EPSILON:
            action = env.action_space.sample()
        else:
            q_values = model(state_input, training=False)
            action = np.argmax(q_values[0])

        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        # Mecanismo Ativo Anti-Travamento:
        if reward > 0:
            last_score_step = steps
            last_reward_total = total_reward

        # Se passar de 40 passos sem comer nada, ela provavelmente travou na parede
        if (steps - last_score_step) > 40:
            reward -= 2.0  # Pune levemente a inércia para ela odiar ficar parada
            if random.random() < 0.4:  # Força 40% de chance de mudar de direção na marra
                action = env.action_space.sample()

        total_reward += reward
        next_frame = preprocess_frame(next_state)
        replay_buffer.append((list(state_deque), action, reward, next_frame, done))
        state_deque.append(next_frame)

        # Treinamento com a lógica DOUBLE DQN
        if len(replay_buffer) > BATCH_SIZE and steps % 4 == 0:
            minibatch = random.sample(replay_buffer, BATCH_SIZE)
            states_b = np.array([np.stack(t[0], axis=-1) for t in minibatch], dtype=np.float32)
            actions_b = np.array([t[1] for t in minibatch], dtype=np.int32)
            rewards_b = np.array([t[2] for t in minibatch], dtype=np.float32)

            next_states_b = []
            for t in minibatch:
                hist = t[0][1:] + [t[3]]
                next_states_b.append(np.stack(hist, axis=-1))
            next_states_b = np.array(next_states_b, dtype=np.float32)
            dones_b = np.array([t[4] for t in minibatch], dtype=np.float32)

            main_next_q = model(next_states_b, training=False).numpy()
            best_actions = np.argmax(main_next_q, axis=1)

            target_next_q = target_model(next_states_b, training=False).numpy()

            targets = model(states_b, training=False).numpy()
            for i in range(BATCH_SIZE):
                if dones_b[i]:
                    targets[i][actions_b[i]] = rewards_b[i]
                else:
                    targets[i][actions_b[i]] = rewards_b[i] + GAMMA * target_next_q[i][best_actions[i]]

            train_step(states_b, targets)

    # Decaimento do Epsilon
    if EPSILON > EPSILON_MIN:
        EPSILON *= EPSILON_DECAY

    # Sincroniza a Rede Alvo a cada 10 episódios
    if episode % 10 == 0:
        target_model.set_weights(model.get_weights())

    # SALVA NO HISTÓRICO ANTES DE IMPRIMIR
    historico_rewards.append(total_reward)
    print(f"Episódio {episode}/{EPISODES} | Recompensa Total={total_reward:.1f} | Epsilon={EPSILON:.4f}")

    if episode % 5 == 0:
        gc.collect()

    if episode % 50 == 0:
        model.save_weights(DRIVE_PATH)
        print(f"\n--- [SEGURANÇA] Novo Checkpoint Dueling Corrigido salvo no Drive! ---\n")
        tf.keras.backend.clear_session()
        gc.collect()

env.close()

# ==========================================
# --- BLOCO DE PLOTAGEM DOS GRÁFICOS ---
# ==========================================
print("\n--- [SISTEMA] GERANDO RELATÓRIO VISUAL DO TREINAMENTO ---")

episodios_eixo = range(EPISODE_START, EPISODE_START + len(historico_rewards))

plt.figure(figsize=(14, 6))

# Gráfico 1: Recompensas Individuais e Média Móvel
plt.subplot(1, 2, 1)
plt.plot(episodios_eixo, historico_rewards, color="royalblue", alpha=0.4, label="Recompensa por Ep.")
if len(historico_rewards) >= 50:
    media_movel = np.convolve(historico_rewards, np.ones(50)/50, mode='valid')
    plt.plot(range(EPISODE_START + 49, EPISODE_START + len(historico_rewards)), media_movel, color="darkorange", linewidth=2, label="Média Móvel (50 eps)")
plt.xlabel("Episódios")
plt.ylabel("Reward")
plt.title("Performance por Episódio")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()

# Gráfico 2: Somatório Acumulado das Recompensas
plt.subplot(1, 2, 2)
somatorio_acumulado = np.cumsum(historico_rewards)
plt.plot(episodios_eixo, somatorio_acumulado, color="crimson", linewidth=2.5, label="Soma Acumulada")
plt.xlabel("Episódios")
plt.ylabel("Recompensa Total Acumulada")
plt.title("Somatório Geral das Recompensas (Curva de Lucro)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()

plt.tight_layout()
plt.show()

# Print estatístico final
if len(historico_rewards) >= 50:
    print(f"\nMédia dos últimos 50 episódios: {np.mean(historico_rewards[-50:]):.2f}")

--- [SISTEMA] CONECTANDO AO GOOGLE DRIVE ---
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- [SUCESSO] Pesos Dueling carregados do Drive! ---
--- [TREINO] Iniciando a Era de Elite (Dueling Double DQN) ---
Episódio 3100/6000 | Recompensa Total=2130.0 | Epsilon=0.2000

--- [SEGURANÇA] Novo Checkpoint Dueling Corrigido salvo no Drive! ---

Episódio 3101/6000 | Recompensa Total=340.0 | Epsilon=0.2000
Episódio 3102/6000 | Recompensa Total=424.0 | Epsilon=0.2000
Episódio 3103/6000 | Recompensa Total=598.0 | Epsilon=0.2000
Episódio 3104/6000 | Recompensa Total=60.0 | Epsilon=0.2000
Episódio 3105/6000 | Recompensa Total=84.0 | Epsilon=0.2000
Episódio 3106/6000 | Recompensa Total=462.0 | Epsilon=0.2000
Episódio 3107/6000 | Recompensa Total=530.0 | Epsilon=0.2000
Episódio 3108/6000 | Recompensa Total=574.0 | Epsilon=0.2000
Episódio 3109/6000 | Recompensa Total=414.0 | Epsilon=0.2000
Episódio 3110/6000 | Recompen

KeyboardInterrupt: 